## Plan de travail
- **Partie 1 :** implémentation d"une couche GAT single-head en NumPy (Exos 1.1 → 1.4 + Quiz 1.1).
- **Partie 2 :** architecture GAT PyG sur Cora, entraînement + Quiz 2.1.
- **Partie 3 :** extraction et visualisation des poids d"attention.
- **Partie 4 :** comparaison GCN / GAT / GATv2 (perf & temps).
- **Annexes :** rapport synthétique + section sur l"usage du GenAI.

In [1]:
# Chargement des dépendances principales et du dataset Cora pour PyG
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

torch.manual_seed(42)
np.random.seed(42)
DATA_ROOT = "Cora"
dataset = Planetoid(root=DATA_ROOT, name="Cora", transform=NormalizeFeatures())
data = dataset[0]
print(dataset)
print(data)

Cora()
Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


Processing...
Done!
c:\Users\asus-\Projets\GNN\.venv\Lib\site-packages\torch_geometric\datasets\planetoid.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, sel

## Partie 1 · NumPy GAT (Exos 1.1 → 1.4)
Compléter progressivement la classe `GATLayerNumPy` ci-dessous : initialisation Xavier, LeakyReLU, coefficients d"attention masqués par l"adjacence et passage avant complet. Chaque TODO renvoie au numéro d"exercice correspondant. Pensez à documenter les réponses aux quiz directement sous les cellules concernées.

In [ ]:
import math

class GATLayerNumPy:
    """Couche GAT single-head en NumPy utilisée dans la Partie 1.

    Les méthodes sont à compléter suivant les exercices décrits dans 3-GAT.md.
    """

    def __init__(self, in_features: int, out_features: int, alpha: float = 0.2):
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        limit = math.sqrt(6.0 / (in_features + out_features))
        rng = np.random.default_rng(42)
        self.W = rng.uniform(-limit, limit, size=(in_features, out_features))  # Exo 1.1
        self.a = rng.uniform(-limit, limit, size=(2 * out_features, 1))        # Exo 1.1

    @staticmethod
    def leaky_relu(x, negative_slope: float = 0.2):
        """Exo 1.2 : implémenter la variante NumPy du LeakyReLU."""
        return np.where(x >= 0, x, negative_slope * x)

    @staticmethod
    def masked_softmax(logits: np.ndarray, mask: np.ndarray):
        """Utilitaire pour appliquer un softmax stable en ne gardant que les voisins (mask = adj)."""
        masked_logits = np.where(mask, logits, -1e9)
        shifted = masked_logits - masked_logits.max(axis=1, keepdims=True)
        exp_scores = np.exp(shifted) * mask
        denom = exp_scores.sum(axis=1, keepdims=True) + 1e-16
        return exp_scores / denom

    def compute_attention_coefficients(self, h: np.ndarray, adj: np.ndarray):
        """Exo 1.3 : calculer e_ij, appliquer LeakyReLU, masquer avec adj, puis normaliser."""
        a_l = self.a[: self.out_features]
        a_r = self.a[self.out_features :]
        f_i = h @ a_l  # (N, 1)
        f_j = h @ a_r  # (N, 1)
        logits = f_i + f_j.T
        logits = self.leaky_relu(logits, self.alpha)
        mask = adj.astype(bool)
        return self.masked_softmax(logits, mask)

    def forward(self, features: np.ndarray, adj: np.ndarray, activation=np.tanh):
        """Exo 1.4 : propager les features via l'aggrégation attentionnelle puis activer."""
        h = features @ self.W
        attention = self.compute_attention_coefficients(h, adj)
        aggregated = attention @ h
        return activation(aggregated) if activation is not None else aggregated

## Partie 2 · Pipeline PyG GAT
Définir ensuite l"architecture `GAT` (2 couches GATConv), la fonction `train()` et les métriques de validation / test. Utiliser les blocs ci-dessous pour structurer l"implémentation avant de passer à la visualisation des poids (Partie 3).

> Astuce : préparez aussi une cellule dédiée aux hyperparamètres pour faciliter les comparaisons ultérieures (Partie 4).

In [ ]:
class GAT(nn.Module):
    """Exercices 2.1-2.2 : architecture GAT PyG (2 couches)."""

    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int,
                 heads: int = 8, dropout: float = 0.6):
        super().__init__()
        self.dropout = dropout
        raise NotImplementedError("Compléter l'initialisation (Exercice 2.1).")

    def forward(self, data):
        """Exercice 2.2 : passe avant avec dropout + GATConv + log_softmax."""
        raise NotImplementedError("Compléter la passe avant (Exercice 2.2).")


def train(model: nn.Module, data, optimizer: torch.optim.Optimizer):
    """Exercice 2.3 : boucle d'entraînement sur les masques d'apprentissage."""
    raise NotImplementedError("Compléter la fonction d'entraînement (Exercice 2.3).")


def evaluate(model: nn.Module, data, mask: torch.Tensor):
    """Utilitaire validation/test pour centraliser l'évaluation des métriques."""
    raise NotImplementedError("Compléter evaluate pour valider/tester les performances.")

## Partie 3 · Visualisation des poids d'attention
Préparer les structures nécessaires pour extraire, analyser et visualiser les poids d'attention (Exos 3.1 → 3.3).
Chaque sous-exercice disposera d'une méthode dédiée dans une classe utilitaire pour garder la logique organisée.

In [ ]:
class AttentionInspector:
    """Exercices 3.1-3.3 : extraction, analyse et visualisation des poids d'attention."""


    def __init__(self, model: nn.Module):
        self.model = model
        self.cached_attention = None  # Contiendra les poids récupérés (Exercice 3.1)


    def extract_attention_weights(self, data, layer_idx: int = 0):
        """Exercice 3.1 : utiliser return_attention_weights=True pour extraire les poids."""
        raise NotImplementedError("Compléter l'extraction des poids d'attention (Exercice 3.1).")


    def describe_attention_statistics(self, attention_weights):
        """Exercice 3.2 : calculer statistiques (moyenne/écart-type) par tête."""
        raise NotImplementedError("Compléter l'analyse statistique des poids (Exercice 3.2).")


    def visualize_attention(self, data, node_idx: int, head: int = 0):
        """Exercice 3.3 : produire une visualisation des poids pour un nœud donné."""
        raise NotImplementedError("Compléter la visualisation des poids (Exercice 3.3).")

## Partie 4 · Comparaison GCN / GAT / GATv2
Structurer les expériences de comparaison (perf, temps, analyse des poids) en définissant une classe de pilotage dédiée aux Exos de la Partie 4.

In [ ]:
class ModelComparisonSuite:
    """Partie 4 : comparer GCN, GAT et GATv2 sur plusieurs jeux de données."""


    def __init__(self, datasets: dict, device: torch.device | None = None):
        self.datasets = datasets
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.results = []
        raise NotImplementedError("Initialiser les structures d'expérimentation (Partie 4).")


    def configure_run(self, model_name: str, hyperparams: dict):
        """Préparer une expérimentation (choix modèle + hyperparamètres)."""
        raise NotImplementedError("Configurer un run (Partie 4).")


    def benchmark(self):
        """Lancer les entraînements, collecter perf/temps/attention selon les exercices."""
        raise NotImplementedError("Implémenter la boucle de comparaison (Partie 4).")


    def summarize(self):
        """Retourner un tableau/rapport synthétique des métriques collectées."""
        raise NotImplementedError("Synthétiser les résultats (Partie 4).")